# Data Exploration and Preparation
<div style="background-color:#f0f4f8; padding:15px; border-left:6px solid #2f5597; border-radius:6px;">

## Purpose of this notebook

This notebook prepares the ZüriWieNeu dataset for the following research-question notebooks.

The main goal is not to answer a research question yet, but to understand the structure of the raw data, inspect important columns, check data types, clean the dataset, and save a processed version for later use.

The cleaned dataset will then be used in the later notebooks:

- `01_R1.ipynb`: Reporting volume over time
- `02_R2.ipynb`: Change in report categories over time
- `03_R3.ipynb`: Spatial distribution of reports in 2025
- `04_R4.ipynb`: Category concentration by Quartier in 2025

### Loading the raw data

The raw ZüriWieNeu CSV file is loaded from the `data/raw` folder using a reusable loading function from the `src` folder.

Separating the loading logic into a function makes the workflow easier to reuse across different notebooks. It also supports the project goal of building a clear and reproducible data-processing workflow.

In [1]:
import sys
sys.path.append("..")
from src.loading import load_csv_data

In [2]:
df_raw = load_csv_data("/Users/laumagoldmann/Desktop/SDS210_IndividualProject/data/raw/stzh.zwn_meldungen_p.csv")

In [8]:
df_raw.head()

,objectid,service_request_id,requested_datetime,agency_sent_datetime,updated_datetime,e,n,service_code,service_name,status,userid,title,detail,media_url,interface_used,service_notice,description,url,geometry
0,1,1,2013-03-14T15:16:15,2013-04-04T07:25:05,2013-04-12T07:59:30,2678968,1247548,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Asp,Auf dem Asphalt des Bürgersteigs hat es eine E...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,https://www.zueriwieneu.ch/report/1,POINT (2678968 1247548)
1,2,2,2013-03-14T15:17:57,2013-03-26T14:05:05,2013-04-12T08:00:22,2680746,1249916,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Vermessungs,Vermessungspunkt ist nicht mehr bündig mit dem...,NaN,Web interface,Diese Reparatur wird von uns in den kommenden ...,Vermessungs: Vermessungspunkt ist nicht mehr b...,https://www.zueriwieneu.ch/report/2,POINT (2680746 1249916)
2,3,4,2013-03-15T09:14:16,2013-03-15T09:55:05,2013-04-12T08:08:10,2684605,1251431,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Beim Trotto,Beim Trottoir sind einige Randsteine defekt un...,https://www.zueriwieneu.ch/photo/4.0.jpeg?bfbb...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Beim Trotto: Beim Trottoir sind einige Randste...,https://www.zueriwieneu.ch/report/4,POINT (2684605 1251431)
3,4,5,2013-03-15T09:17:15,2013-03-20T10:05:05,2013-04-12T08:09:05,2681754,1250376,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,fixed - council,16624,Auf dem Par,Auf dem Parkplatz beim Waidspital sind einige ...,https://www.zueriwieneu.ch/photo/5.0.jpeg?e309...,Web interface,Diese Reparatur wird von uns in den kommenden ...,Auf dem Par: Auf dem Parkplatz beim Waidspital...,https://www.zueriwieneu.ch/report/5,POINT (2681754 1250376)
4,5,6,2013-03-15T10:36:53,2013-04-22T18:25:05,2013-04-23T13:50:33,2683094,1247762,Abfall/Sammelstelle,Abfall/Sammelstelle,fixed - council,16624,Arbeitskist,Arbeitskiste ist rund herum verschmiert,https://www.zueriwieneu.ch/photo/6.0.jpeg?8e65...,Web interface,Dieses Graffiti wird von uns in den kommenden ...,Arbeitskist: Arbeitskiste ist rund herum versc...,https://www.zueriwieneu.ch/report/6,POINT (2683094 1247762)


### Initial inspection of the dataset

The first inspection of the raw dataset shows the available columns, data types, and general structure of the ZüriWieNeu data.

Important columns for this project include:

- `service_request_id`: unique identifier for each report
- `requested_datetime`: date and time when the report was submitted
- `updated_datetime`: date and time when the report was last updated
- `e` and `n`: Swiss coordinate values used for spatial analysis
- `service_name`: report category
- `status`: current status of the report
- `title`, `detail`, and `description`: textual information about the report
- `interface_used`: information about how the report was submitted

The column `requested_datetime` is especially important for the temporal analyses, while `e` and `n` are needed later for spatial analysis.

### Data cleaning workflow

The dataset is cleaned using reusable functions from `src/cleaning.py`.

The cleaning workflow includes three main steps:

1. Convert date columns from text to datetime format.
2. Remove duplicate reports based on `service_request_id`.
3. Add time-based columns such as `year`, `month`, and `year_month`.

These steps make the dataset suitable for temporal analysis and prepare it for later spatial analysis.

In [3]:
from src.cleaning import parse_datetime_columns, remove_duplicate_reports

# using the functions from the cleaning.py script for changing the time and to check for duplicates
df_test = parse_datetime_columns(df_raw)
df_test = remove_duplicate_reports(df_test)

# output shows if there is a difference to the raw dataset and the cleaned dataset
print(df_raw.shape)
print(df_test.shape)

(72567, 19)
(72567, 19)


### Duplicate check

Duplicate reports were checked using the `service_request_id` column, which identifies individual ZüriWieNeu reports.

The row count before and after removing duplicates was the same. This means that no duplicate report IDs were found in the dataset.

This is useful for the later analysis because each row can be treated as one unique report.

In [4]:
import src.cleaning

from src.cleaning import (
    parse_datetime_columns,
    remove_duplicate_reports,
    add_time_columns
)

df_test = parse_datetime_columns(df_raw)
df_test = remove_duplicate_reports(df_test)
df_test = add_time_columns(df_test)

df_test[["requested_datetime", "year", "month", "year_month"]].head()


,requested_datetime,year,month,year_month
0,2013-03-14 15:16:15,2013,3,2013-03-01
1,2013-03-14 15:17:57,2013,3,2013-03-01
2,2013-03-15 09:14:16,2013,3,2013-03-01
3,2013-03-15 09:17:15,2013,3,2013-03-01
4,2013-03-15 10:36:53,2013,3,2013-03-01


### Added time columns

To support the temporal analyses in the later notebooks, additional time-based columns were created from `requested_datetime`.

The added columns are:

- `year`: the calendar year of the report
- `month`: the calendar month of the report
- `year_month`: the month of the report represented as a datetime value

The `year_month` column is represented by the first day of the month, for example `2013-03-01`. This does not mean that the report was submitted on that exact day. Instead, it represents the month March 2013 and is useful for plotting monthly time series.

### Combining cleaning steps into one reusable function

Combining cleaning steps into one reusable functionAfter defining the individual cleaning functions, they are combined into one main cleaning function called `clean_reports()`.

This function applies the full cleaning workflow in a fixed order:

1. Parse datetime columns.
2. Remove duplicate reports.
3. Add time-based columns for analysis.

Combining these steps into one function makes the workflow easier to reuse across all research-question notebooks. Instead of repeating several cleaning commands in every notebook, the raw dataset can be cleaned with one clear function call:

```python
df_clean = clean_reports(df_raw)



In [5]:
import importlib
import src.cleaning

importlib.reload(src.cleaning)

from src.cleaning import clean_reports

df_clean = clean_reports(df_raw)

df_clean[["requested_datetime", "year", "month", "year_month"]].head()

,requested_datetime,year,month,year_month
0,2013-03-14 15:16:15,2013,3,2013-03-01
1,2013-03-14 15:17:57,2013,3,2013-03-01
2,2013-03-15 09:14:16,2013,3,2013-03-01
3,2013-03-15 09:17:15,2013,3,2013-03-01
4,2013-03-15 10:36:53,2013,3,2013-03-01


### Saving the processed dataset

After cleaning, the processed dataset is saved to the `data/processed` folder.

Saving a cleaned version of the dataset makes it possible to reuse the prepared data later without repeating every exploratory step. At the same time, the research-question notebooks still use the reusable cleaning functions, so the full workflow remains reproducible from the raw data.

In [7]:
df_clean.to_csv("../data/processed/zueriwieneu_cleaned.csv", index=False)

<div style="background-color:#eef9f0; padding:15px; border-left:6px solid #3c9d5d; border-radius:6px;">

## Conclusion

This notebook explored and prepared the raw ZüriWieNeu dataset for further analysis.

The main preparation steps were:

- loading the raw CSV data
- inspecting columns and data types
- converting date columns to datetime format
- checking for duplicate reports
- adding time-based columns for temporal analysis
- saving a processed version of the cleaned dataset

The cleaned data is now ready to be used in the following research-question notebooks. The next notebooks will analyse reporting trends over time, changes in report categories, and spatial patterns across Zurich Quartiere.